# CXR Essential-Tag Evaluator — 실행 가이드 (한글)

X-ray **essential tag** 기준으로 DICOM 메타데이터 품질을 **series 단위**로 평가합니다.
**`df_input`(아래 스키마)이 이미 준비되어 있다고 가정**하고, 로드 → 평가 → 저장까지 진행합니다.

- 개념/지표 설명: `CxrEssentialTagEvaluator_guide_ko.md`
- 평가기 코드: `Evaluator/CxrEssentialTagEvaluator.py`

> 이 노트북은 저장소의 `DicomStandardEvaluator/` 폴더에서 실행하는 것을 기준으로 경로가 잡혀 있습니다.


## 0. 준비

`df_input` **필수 컬럼** (long-format: 한 행 = 하나의 instance × 하나의 Tag × Value)

| 컬럼 | DICOM | 설명 |
|---|---|---|
| `IOD` | (0008,0016) | SOP Class UID 기반 IOD (참고용) |
| `study_instance_uid` | (0020,000D) | Study Instance UID |
| `series_instance_uid` | (0020,000E) | Series Instance UID — **집계 기준 키** |
| `Manufacturer` | (0008,0070) | 그룹 분석용(옵션) |
| `ScannerModel` | (0008,1090) | 그룹 분석용(옵션) |
| `Tag` | | DICOM Tag (`'00180015'` 권장) |
| `AttributeName` | | Attribute Name (옵션) |
| `Value` | | Tag value (문자열) |


In [1]:
import os
import sys
import numpy as np
import pandas as pd

REF_XLSX   = '../files/CxrEssentialTags/CxrEssentialTags_ReferenceSet.xlsx'
EVAL_DIR   = 'Evaluator'
OUTPUT_DIR = 'output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

sys.path.append(EVAL_DIR)
from CxrEssentialTagEvaluator import CxrEssentialTagEvaluator

REQUIRED_COLS = ['IOD', 'study_instance_uid', 'series_instance_uid',
                 'Manufacturer', 'ScannerModel', 'Tag', 'AttributeName', 'Value']

## 1. `df_input` 로드

**여기에 본인 데이터를 넣으세요.** 아래 셀에서 `df_input`을 준비합니다.
- 실제 사용: `df_input = pd.read_parquet('내데이터.parquet')` 로 교체
- 지금은 노트북이 끝까지 도는지 확인할 수 있도록 **작은 예시(dummy)** 를 제공합니다.


In [2]:
# ────────────────────────────────────────────────────────────────
# 실제 사용 시: 아래 한 줄로 교체하고 dummy 블록은 지우세요.
#   df_input = pd.read_parquet('/path/to/your_df_input.parquet')
# ────────────────────────────────────────────────────────────────

# --- (예시 dummy) 3 series, 실제 스키마와 동일 --------------------
_rows = []
def _r(se, tag, attr, val, iod='DX', st='ST1', mfr='GE HealthCare', mdl='ModelX'):
    _rows.append(dict(IOD=iod, study_instance_uid=st, series_instance_uid=se,
                      Manufacturer=mfr, ScannerModel=mdl, Tag=tag,
                      AttributeName=attr, Value=val))
# series SE1: 인스턴스1 'chest'(정확), 인스턴스2 'port chest'(부분포함) → series 준수
_r('SE1', '00180015', 'Body Part Examined', 'chest')
_r('SE1', '00180015', 'Body Part Examined', 'port chest')
_r('SE1', '00080060', 'Modality', 'DX')
_r('SE1', '00185101', 'View Position', 'PA')
# series SE2: Modality 'XR'(미포함), View Position 'LATERAL'(미포함)
_r('SE2', '00080060', 'Modality', 'XR', st='ST2', mfr='Philips')
_r('SE2', '00185101', 'View Position', 'LATERAL', st='ST2', mfr='Philips')
_r('SE2', '00180015', 'Body Part Examined', 'CHEST', st='ST2', mfr='Philips')
# series SE3: Modality 값 없음(빈 문자열)
_r('SE3', '00080060', 'Modality', '', st='ST3', mfr='Philips')
_r('SE3', '00280004', 'Photometric Interpretation', 'MONOCHROME2', st='ST3', mfr='Philips')

df_input = pd.DataFrame(_rows)
# -----------------------------------------------------------------

df_input.head()

,IOD,study_instance_uid,series_instance_uid,Manufacturer,ScannerModel,Tag,AttributeName,Value
0,DX,ST1,SE1,GE HealthCare,ModelX,00180015,Body Part Examined,chest
1,DX,ST1,SE1,GE HealthCare,ModelX,00180015,Body Part Examined,port chest
2,DX,ST1,SE1,GE HealthCare,ModelX,00080060,Modality,DX
3,DX,ST1,SE1,GE HealthCare,ModelX,00185101,View Position,PA
4,DX,ST2,SE2,Philips,ModelX,00080060,Modality,XR


### 1-1. 스키마 검증

In [3]:
missing = [c for c in REQUIRED_COLS if c not in df_input.columns]
assert not missing, f'df_input에 다음 컬럼이 없습니다: {missing}'
print('컬럼 OK:', REQUIRED_COLS)
print('rows   :', len(df_input))
print('studies:', df_input['study_instance_uid'].nunique(),
      '| series:', df_input['series_instance_uid'].nunique())

컬럼 OK: ['IOD', 'study_instance_uid', 'series_instance_uid', 'Manufacturer', 'ScannerModel', 'Tag', 'AttributeName', 'Value']
rows   : 9
studies: 3 | series: 3


## 2. 표준(essential tags + CS 허용값) 로드

In [4]:
df_standard = pd.read_excel(REF_XLSX)
print('essential tags:', len(df_standard), '| CS tags:', (df_standard['VR'] == 'CS').sum())
df_standard[['Tag', 'Attribute Name', 'VR', 'CS_allowable_values']].head(10)

essential tags: 28 | CS tags: 9


,Tag,Attribute Name,VR,CS_allowable_values
0,180015,Body Part Examined,CS,"['3RDVENTRICLE', '4THVENTRICLE', 'ABDOMEN', 'A..."
1,185101,View Position,CS,"['AP', 'LL', 'LLD', 'LLO', 'PA', 'RL', 'RLD', ..."
2,185100,Patient Position,CS,"['AFDL', 'AFDR', 'AFP', 'AFS', 'FFDL', 'FFDR',..."
3,80060,Modality,CS,"['ANN', 'AR', 'ASMT', 'AU', 'BDUS', 'BI', 'BMD..."
4,200062,Image Laterality,CS,"['B', 'L', 'R', 'U']"
5,282110,Lossy Image Compression,CS,"['00', '01']"
6,187060,Exposure Control Mode,CS,"['AUTOMATIC', 'MANUAL']"
7,187004,Detector Type,CS,"['DIRECT', 'FILM', 'SCINTILLATOR', 'STORAGE']"
8,280004,Photometric Interpretation,CS,"['MONOCHROME1', 'MONOCHROME2']"
9,280010,Rows,US,[]


## 3. Evaluator 생성

In [5]:
evaluator = CxrEssentialTagEvaluator(df_input, df_standard)

## 4. 평가

### 4-1) 전체 데이터셋 (series-level)
태그별 `tag_completeness`, `value_completeness`, `value_conformance`.

In [6]:
overall = evaluator.analyze(group_cols=None)
overall[['Tag', 'Attribute Name', 'VR', 'total_series',
         'series_with_tag', 'series_with_value', 'series_conform',
         'tag_completeness', 'value_completeness', 'value_conformance']]

,Tag,Attribute Name,VR,total_series,series_with_tag,series_with_value,series_conform,tag_completeness,value_completeness,value_conformance
0,00180015,Body Part Examined,CS,3,2,2,2,0.666667,1.000000,1.0
1,00185101,View Position,CS,3,2,2,1,0.666667,1.000000,0.5
2,00185100,Patient Position,CS,3,0,0,0,0.000000,NaN,NaN
3,00080060,Modality,CS,3,3,2,1,1.000000,0.666667,0.5
4,00200062,Image Laterality,CS,3,0,0,0,0.000000,NaN,NaN
5,00282110,Lossy Image Compression,CS,3,0,0,0,0.000000,NaN,NaN
6,00187060,Exposure Control Mode,CS,3,0,0,0,0.000000,NaN,NaN
7,00187004,Detector Type,CS,3,0,0,0,0.000000,NaN,NaN
8,00280004,Photometric Interpretation,CS,3,1,1,1,0.333333,1.000000,1.0
9,00280010,Rows,US,3,0,0,0,0.000000,NaN,NaN


### 4-2) 그룹별 + 이질성 통계

`group_cols`(예: `['Manufacturer']`, `['Manufacturer','ScannerModel']`)별 지표와,
그룹 간 **Mean/Std/CV(%)/Range** 통계를 얻습니다.
> 데이터가 비식별화되어 Manufacturer 값이 비어 있으면 그룹이 1개가 되어 통계가 자명해집니다.

In [7]:
rates, stats = evaluator.analyze_with_stats(group_cols=['Manufacturer'])
stats.head(15)

,Tag,Attribute Name,VR,Metric,Mean,Std,CV(%),Min,Max,Range,n_groups,n_groups_valid
0,00080060,Modality,CS,tag_completeness,1.00,0.000000,0.000000,1.0,1.0,0.0,2,2
1,00080060,Modality,CS,value_completeness,0.75,0.353553,47.140452,0.5,1.0,0.5,2,2
2,00080060,Modality,CS,value_conformance,0.50,0.707107,141.421356,0.0,1.0,1.0,2,2
3,00080070,Manufacturer,LO,tag_completeness,0.00,0.000000,NaN,0.0,0.0,0.0,2,2
4,00080070,Manufacturer,LO,value_completeness,NaN,NaN,NaN,NaN,NaN,NaN,2,0
5,00080070,Manufacturer,LO,value_conformance,NaN,NaN,NaN,NaN,NaN,NaN,2,0
6,00081090,Manufacturer's Model Name,LO,tag_completeness,0.00,0.000000,NaN,0.0,0.0,0.0,2,2
7,00081090,Manufacturer's Model Name,LO,value_completeness,NaN,NaN,NaN,NaN,NaN,NaN,2,0
8,00081090,Manufacturer's Model Name,LO,value_conformance,NaN,NaN,NaN,NaN,NaN,NaN,2,0
9,00180015,Body Part Examined,CS,tag_completeness,0.75,0.353553,47.140452,0.5,1.0,0.5,2,2


### 4-3) Conformance 세부 리포트 (CS 태그)

정확 일치는 아니지만 허용값을 **포함**하는 값(**partial**, 예: `CHEST`↔`port chest`)과
허용값을 **전혀 포함하지 않는** 값(**none**, 예: `XR`/`LATERAL`)을 분리합니다.
대-소문자는 pass/fail과 무관합니다.

In [8]:
report = evaluator.conformance_subreport(group_cols=None)
report['summary']

category,Tag,Attribute Name,n_values,n_pass,n_partial,n_none,pct_pass,pct_partial,pct_none
0,"(0008,0060)",Modality,2,1,0,1,50.00,0.00,50.0
1,"(0018,5101)",View Position,2,1,0,1,50.00,0.00,50.0
2,"(0018,0015)",Body Part Examined,3,2,1,0,66.67,33.33,0.0
3,"(0028,0004)",Photometric Interpretation,1,1,0,0,100.00,0.00,0.0


In [9]:
# partial: 허용값을 포함하지만 정확히 일치하지 않는 값 (건수 + value_counts)
report['partial'].drop(columns=['Defined Values'])

,Tag,Attribute Name,Value,count,n_series,pct
0,"(0018,0015)",Body Part Examined,port chest,1,1,33.33


In [10]:
# none: 허용값을 전혀 포함하지 않는 값 (건수 + % + value_counts)
report['none'].drop(columns=['Defined Values'])

,Tag,Attribute Name,Value,count,n_series,pct
0,"(0008,0060)",Modality,XR,1,1,50.0
1,"(0018,5101)",View Position,LATERAL,1,1,50.0


## 5. 저장 (workspace/output)

In [11]:
# 태그별 completeness/conformance
overall.to_excel(os.path.join(OUTPUT_DIR, 'cxr_overall_rates.xlsx'), index=False)

# 그룹별 지표 + 이질성 통계
rates.to_excel(os.path.join(OUTPUT_DIR, 'cxr_rates_by_manufacturer.xlsx'), index=False)
stats.to_excel(os.path.join(OUTPUT_DIR, 'cxr_stats_by_manufacturer.xlsx'), index=False)

# conformance 리포트 3종: cxr_conformance_{summary,partial,none}.csv
evaluator.export_conformance_subreport(os.path.join(OUTPUT_DIR, 'cxr_conformance'),
                                       group_cols=None)
# (요약) 미준수 unique value 통합 CSV
evaluator.export_unconformed_values(os.path.join(OUTPUT_DIR, 'cxr_unconformed_values.csv'))

print('저장 완료:', sorted(os.listdir(OUTPUT_DIR)))

저장 완료: ['cxr_conformance_none.csv', 'cxr_conformance_partial.csv', 'cxr_conformance_summary.csv', 'cxr_overall_rates.xlsx', 'cxr_rates_by_manufacturer.xlsx', 'cxr_stats_by_manufacturer.xlsx', 'cxr_unconformed_values.csv']


## 6. 해석 팁

- `tag_completeness` ↓ : 해당 태그가 아예 빠진 series가 많음 (파티션/장비 미기록 등)
- `value_completeness` ↓ : 태그는 있으나 값이 빈 series가 많음 (비식별화·공란)
- `value_conformance` ↓ : CS 값이 표준용어를 벗어남 → `report['partial']`/`report['none']`에서 실제 값 확인
  - **partial**: 표준값을 포함한 변형(예: `PORT CHEST`) → 매핑/정규화로 구제 가능
  - **none**: 표준에 아예 없는 값(예: `LATERAL`) → 코드 표준화 필요
- 표준 허용값을 바꾸려면 `CxrEssentialTags_ReferenceSet.xlsx`의 `CS_allowable_values` 수정 또는
  `DicomStandardRetrieval/build_cxr_essential_reference.py`로 재생성.
